### https://www.kaggle.com/competitions/drawing-with-llms

In [8]:
import kagglehub
import pandas as pd

In [9]:
import concurrent
import io
import logging
import re
import re2

import cairosvg
import kagglehub
import torch
from lxml import etree
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

svg_constraints = kagglehub.package_import('metric/svg-constraints')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('DEVICE', DEVICE)

class SVGSanitizer:
    def __init__(self, constraints, default_svg):
        self.constraints = constraints
        self.default_svg = default_svg
    
    def enforce_constraints(self, svg_string: str) -> str:
        """Enforces constraints on an SVG string, removing disallowed elements
        and attributes.

        Parameters
        ----------
        svg_string : str
            The SVG string to process.

        Returns
        -------
        str
            The processed SVG string, or the default SVG if constraints
            cannot be satisfied.
        """
        logging.info('Sanitizing SVG...')

        try:
            parser = etree.XMLParser(remove_blank_text=True, remove_comments=True)
            root = etree.fromstring(svg_string, parser=parser)
        except etree.ParseError as e:
            logging.error('SVG Parse Error: %s. Returning default SVG.', e)
            return self.default_svg
    
        elements_to_remove = []
        for element in root.iter():
            try:
                tag_name = etree.QName(element.tag).localname
            except ValueError as e:
                return self.default_svg
    
            # Remove disallowed elements
            if tag_name not in self.constraints.allowed_elements:
                elements_to_remove.append(element)
                continue  # Skip attribute checks for removed elements
    
            # Remove disallowed attributes
            attrs_to_remove = []
            for attr in element.attrib:
                attr_name = etree.QName(attr).localname
                if (
                    attr_name
                    not in self.constraints.allowed_elements[tag_name]
                    and attr_name
                    not in self.constraints.allowed_elements['common']
                ):
                    attrs_to_remove.append(attr)
    
            for attr in attrs_to_remove:
                logging.debug(
                    'Attribute "%s" for element "%s" not allowed. Removing.',
                    attr,
                    tag_name,
                )
                del element.attrib[attr]
    
            # Check and remove invalid href attributes
            for attr, value in element.attrib.items():
                 if etree.QName(attr).localname == 'href' and not value.startswith('#'):
                    logging.debug(
                        'Removing invalid href attribute in element "%s".', tag_name
                    )
                    del element.attrib[attr]

            # Validate path elements to help ensure SVG conversion
            if tag_name == 'path':
                d_attribute = element.get('d')
                if not d_attribute:
                    logging.warning('Path element is missing "d" attribute. Removing path.')
                    elements_to_remove.append(element)
                    continue # Skip further checks for this removed element
                # Use regex to validate 'd' attribute format
                path_regex = re2.compile(
                    r'^'  # Start of string
                    r'(?:'  # Non-capturing group for each command + numbers block
                    r'[MmZzLlHhVvCcSsQqTtAa]'  # Valid SVG path commands (adjusted to exclude extra letters)
                    r'\s*'  # Optional whitespace after command
                    r'(?:'  # Non-capturing group for optional numbers
                    r'-?\d+(?:\.\d+)?(?:[Ee][+-]?\d+)?'  # First number
                    r'(?:[\s,]+-?\d+(?:\.\d+)?(?:[Ee][+-]?\d+)?)*'  # Subsequent numbers with mandatory separator(s)
                    r')?'  # Numbers are optional (e.g. for Z command)
                    r'\s*'  # Optional whitespace after numbers/command block
                    r')+'  # One or more command blocks
                    r'\s*'  # Optional trailing whitespace
                    r'$'  # End of string
                )
                if not path_regex.match(d_attribute):
                    logging.warning(
                        'Path element has malformed "d" attribute format. Removing path.'
                    )
                    elements_to_remove.append(element)
                    continue
                logging.debug('Path element "d" attribute validated (regex check).')
        
        # Remove elements marked for removal
        for element in elements_to_remove:
            if element.getparent() is not None:
                element.getparent().remove(element)
                logging.debug('Removed element: %s', element.tag)

        try:
            cleaned_svg_string = etree.tostring(root, encoding='unicode')
            return cleaned_svg_string
        except ValueError as e:
            logging.error(
                'SVG could not be sanitized to meet constraints: %s', e
            )
            return self.default_svg

class SVGProcessor:
    @staticmethod
    def clean_and_extract_svgs(text, default_svg):
        text = re.sub(r'^.*?(<svg\b)', r'\1', text, flags=re.DOTALL)
        svg_blocks = re.findall(r'<svg\b.*?</svg>', text, re.DOTALL)
    
        if svg_blocks:
            tmp = re.findall(r'<svg\b.*?', svg_blocks[-1], re.DOTALL)
            if len(tmp) > 1:
                tmp2 = svg_blocks[-1].split('<svg')
                return '<svg ' + tmp2[-1]
            else:
                return svg_blocks[-1]
        else:
            if "<svg" in text and "</svg>" not in text:
                text += "</svg>"
                return text
            return default_svg
    
    @staticmethod
    def svg_conversion_check(topic, base_svg_code, default_svg):
        try:
            cairosvg.svg2png(bytestring=base_svg_code.encode('utf-8'), write_to="temp.png")
            return base_svg_code
        except Exception as e:
            print(f"Failed to convert {topic} due to {str(e)}, Returning default SVG.")
            return default_svg


class Model:
    def __init__(self):
        self.model="model"
        
        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)
        
    def clean_svg(self, base_svg_code: str, max_new_tokens=1024) -> str:
        base_svg_code = SVGProcessor.clean_and_extract_svgs(base_svg_code, self.default_svg)
        clean_svg_code = self.sanitizer.enforce_constraints(base_svg_code)
        return clean_svg_code



This code could modify your python environment or operating system.

Review this code at https://www.kaggle.com/code/metric/svg-constraints/versions/1
or in your download cache at /home/vino/.cache/kagglehub/notebooks/metric/svg-constraints/output/versions/1

It is strongly recommended that you run this code within a container
such as Docker to provide a secure, isolated execution environment.
See https://www.kaggle.com/docs/packages for more information.

Do you want to proceed? (y)es/[no]:  y


DEVICE cuda


In [10]:
model=Model()

### Response Loader from CSV batch files

In [5]:
import os
import pandas as pd

# Directory containing CSV files
csv_dir = './batches_async/second_100_batches_gemini_20_flash'

# Get list of all CSV files in the directory
csv_files = [f for f in os.listdir(csv_dir) if f.endswith('.csv')]

# Read and combine them
df_list = [pd.read_csv(os.path.join(csv_dir, file)) for file in csv_files]
combined_df = pd.concat(df_list, ignore_index=True)

df=combined_df.copy()
print(df.shape)

(10000, 3)


In [8]:
import re
def extract_from_gemini_response(text):
    pattern = r'```xml\n(.*?)\n```'
    
    # Search for the pattern
    match = re.search(pattern, text, re.DOTALL)
    
    # Check if a match was found and extract the SVG code
    if match:
        svg_code = match.group(1).strip() # .strip() removes leading/trailing whitespace
        return svg_code
    else:
        print("SVG code block not found in the string.")
        return 0 

In [9]:
from tqdm import tqdm
tqdm.pandas()
df['extracted_svg'] = df['gemini_response_text'].progress_apply(lambda x: extract_from_gemini_response(x))
df=df[df['extracted_svg']!=0]
print(df.shape)

100%|██████████████████████████████████| 10000/10000 [00:00<00:00, 95159.21it/s]

SVG code block not found in the string.
SVG code block not found in the string.
(9998, 4)


In [10]:
df['gemini_response_text'].iloc[0]

'```xml\n<svg width="200" height="200" viewBox="0 0 200 200" fill="none" xmlns="http://www.w3.org/2000/svg">\n  <rect width="200" height="200" fill="#000033"/>\n  <circle cx="30" cy="40" r="2" fill="white"/>\n  <circle cx="150" cy="60" r="1" fill="white"/>\n  <circle cx="80" cy="100" r="3" fill="white"/>\n  <circle cx="180" cy="120" r="1.5" fill="white"/>\n  <circle cx="50" cy="160" r="2.5" fill="white"/>\n  <circle cx="120" cy="30" r="1" fill="white"/>\n  <circle cx="20" cy="100" r="1.5" fill="white"/>\n  <circle cx="160" cy="180" r="2" fill="white"/>\n  <circle cx="90" cy="50" r="1" fill="white"/>\n  <circle cx="40" cy="120" r="2" fill="white"/>\n  <circle cx="110" cy="170" r="1.5" fill="white"/>\n  <circle cx="70" cy="20" r="2" fill="white"/>\n  <circle cx="140" cy="90" r="1" fill="white"/>\n  <circle cx="10" cy="180" r="3" fill="white"/>\n  <circle cx="190" cy="40" r="1.5" fill="white"/>\n  <circle cx="60" cy="70" r="2" fill="white"/>\n  <circle cx="130" cy="140" r="1" fill="white"

### Topic Loader from txt batch files

In [39]:
# import os
# import pandas as pd

# txt_dir = './drawing-with-llms/gemini_20_topics'
# txt_files = [f for f in os.listdir(txt_dir) if f.endswith('.txt')]
# print(f"Found {len(txt_files)} files.")

# all_topics = []

# for file in txt_files:
#     file_path = os.path.join(txt_dir, file)
#     with open(file_path, 'r', encoding='utf-8') as f:
#         lines = f.readlines()
#         # Clean and filter lines
#         topics = [
#             line.strip().strip('"') for line in lines
#             if line.strip()
#             and not any(substr in line for substr in ['```', 'csv', '[', ']','Topic', 'topic'])
#         ]
#         #print(f"file: {file}, lines-topics: {len(lines)}:{len(topics)}")
#         all_topics.extend(topics)

# # Create DataFrame
# df = pd.DataFrame({'description': all_topics})
# print(df.head())
# print(f"Total topics loaded: {len(df)}")
# df=df.drop_duplicates(['description'])
# df.to_csv('tmp.csv',index=False)

Found 198 files.
                                 description
0       a lone birch tree against a pale sky
1   cerulean waves crashing on a sandy beach
2  a vibrant sunflower field under a hot sun
3       geometric shapes in shades of sunset
4       a flowing gown of emerald green silk
Total topics loaded: 62287


In [11]:
import sys
sys.path.append('./utils')
from siglip_class import SVGMetricEvaluator
from aesthetic_evaluator import AestheticEvaluator

In [12]:
#SigLip Score
from tqdm import tqdm
tqdm.pandas()
evaluator = SVGMetricEvaluator()
df['gemini_sl_score'] = df.progress_apply(lambda row: evaluator.svg_metric(row['description'], row['extracted_svg']), axis=1)

  4%|█▋                                      | 427/9998 [00:24<08:25, 18.93it/s]

An error occurred: 


  5%|█▉                                      | 472/9998 [00:27<08:15, 19.21it/s]

An error occurred: Root node is {http://www://www.w3.org/2000/svg}svg. Should be one of marker, svg, image, or g.


  5%|█▉                                      | 477/9998 [00:27<07:46, 20.41it/s]

...

 95%|█████████████████████████████████████▏ | 9534/9998 [09:32<00:24, 19.02it/s]

An error occurred: 


 97%|█████████████████████████████████████▉ | 9727/9998 [09:43<00:14, 18.44it/s]

An error occurred: 


100%|███████████████████████████████████████| 9998/9998 [10:00<00:00, 16.65it/s]
/tmp/ipykernel_423117/649545007.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['gemini_sl_score'] = df.progress_apply(lambda row: evaluator.svg_metric(row['description'], row['extracted_svg']), axis=1)


In [13]:
#Aes Score
from tqdm import tqdm
tqdm.pandas()
aes_eval = AestheticEvaluator()
df['gemini_aes_score'] = df.progress_apply(lambda row: aes_eval.get_score(row['extracted_svg']), axis=1)

  4%|█▋                                      | 425/9998 [00:47<14:44, 10.82it/s]

Errror in scoring: return 0.00


  5%|█▉                                      | 471/9998 [00:53<16:09,  9.83it/s]

Errror in scoring: return 0.00


  5%|█▉                                      | 476/9998 [00:53<13:32, 11.72it/s]

...

 95%|█████████████████████████████████████▏ | 9533/9998 [17:56<00:42, 10.95it/s]

Errror in scoring: return 0.00


 97%|█████████████████████████████████████▉ | 9727/9998 [18:18<00:24, 11.19it/s]

Errror in scoring: return 0.00


100%|███████████████████████████████████████| 9998/9998 [18:50<00:00,  8.84it/s]
/tmp/ipykernel_423117/2198925642.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['gemini_aes_score'] = df.progress_apply(lambda row: aes_eval.get_score(row['extracted_svg']), axis=1)


In [14]:
#combined score
df['combined_score'] = (df['gemini_sl_score']+df['gemini_sl_score']+df['gemini_aes_score'])/3

/tmp/ipykernel_423117/1019658407.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['combined_score'] = (df['gemini_sl_score']+df['gemini_sl_score']+df['gemini_aes_score'])/3


In [15]:
print('Mean SL Score:',df['gemini_sl_score'].mean())

Mean SL Score: 0.32715006968857013


In [16]:
print('Mean AES Score:',df['gemini_aes_score'].mean())

Mean AES Score: 0.4449296654713442


In [17]:
df[df['gemini_sl_score'] >= 0.5].shape

(3219, 7)

In [18]:
df.to_csv('gemini_batch_2_score_master.csv',index=False)
